# EduReport-Summarizer — Complete Pipeline & GitHub Upload

This notebook covers the full pipeline **and** the GitHub upload.

| Step | Script | Task |
|------|--------|------|
| 1 | `01_convert_to_docx.py` | `.doc/.pdf` → `.docx` |
| 2 | `02_docx_to_sections.py` | `.docx` → structured JSONL |
| 3 | `03_make_sft_dataset.py` | JSONL + seeds → SFT dataset |
| 4 | `04_train_sft_qwen_lora.py` | QLoRA fine-tuning *(skip — checkpoint exists)* |
| 5 | `05_batch_infer_mainidea.py` | Batch inference → `.txt` abstracts |
| 6 | *(this notebook)* | GitHub upload |

> **Project root:** `D:\语义分析模型训练\PythonProject`  
> All scripts are in the **root directory** (not in a subfolder).


---
## 0. Configuration
Run this cell first — sets all paths used throughout the notebook.


In [ ]:
from pathlib import Path
import subprocess, sys, os

BASE_DIR    = Path(r'D:\语义分析模型训练\PythonProject')
DATA_DIR    = BASE_DIR / 'data'
SEED_DIR    = BASE_DIR / 'seed'
RESULT_DIR  = BASE_DIR / 'result'
EXAMPLE_DIR = BASE_DIR / 'example'
RUN_NAME    = 'qwen2.5-mainidea-final-stable-fast-infer5-seedguide'
CKPT_DIR    = BASE_DIR / 'checkpoints' / RUN_NAME
LORA_DIR    = CKPT_DIR / 'final_best'

MODEL_NAME  = 'Qwen/Qwen2.5-7B-Instruct'
EMB_MODEL   = 'BAAI/bge-base-zh-v1.5'
SECTION_TITLES = ['研究问题', '核心概念', '研究目标和内容', '研究成果', '研究效果']

print('=== Directory Check ===')
checks = [
    ('BASE_DIR  ', BASE_DIR),
    ('DATA_DIR  ', DATA_DIR),
    ('RESULT_DIR', RESULT_DIR),
    ('EXAMPLE_DIR', EXAMPLE_DIR),
    ('LORA_DIR  ', LORA_DIR),
]
for name, d in checks:
    status = 'OK     ' if d.exists() else 'MISSING'
    print(f'  [{status}] {name} -> {d}')


---
## 1. Environment Check


In [ ]:
import torch

print('=== Package Versions ===')
import transformers, peft, sentence_transformers
print(f'  torch              : {torch.__version__}')
print(f'  transformers       : {transformers.__version__}')
print(f'  peft               : {peft.__version__}')
print(f'  sentence_transformers: {sentence_transformers.__version__}')

print()
print('=== GPU Status ===')
print(f'  CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU            : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  VRAM           : {vram:.1f} GB')
else:
    print('  WARNING: CUDA not available. Steps 4 and 5 require GPU.')


---
## Step 1 — Convert .doc / .pdf to .docx

**Script:** `01_convert_to_docx.py`  
**Input:** `研究报告/` folder  
**Output:** `.docx` files in `data/`

Conversion strategy:
- **Primary:** MS Word COM automation (Windows)
- **Fallback 1:** LibreOffice headless
- **Fallback 2:** `pdf2docx`


In [ ]:
script_01 = BASE_DIR / '01_convert_to_docx.py'
print('Script exists:', script_01.exists())

docx_count = len(list(DATA_DIR.glob('*.docx')))
print(f'Current .docx files in data/: {docx_count}')

# Uncomment to run conversion:
# subprocess.run([sys.executable, str(script_01)], check=True, cwd=str(BASE_DIR))


---
## Step 2 — Parse .docx into Five-Section JSONL

**Script:** `02_docx_to_sections.py`  
**Output:** `data/reports_sections.jsonl`

Each line: `{doc_id, title, sections, full_text}`


In [ ]:
import json

script_02 = BASE_DIR / '02_docx_to_sections.py'
out_jsonl  = DATA_DIR / 'reports_sections.jsonl'

print('Script exists:', script_02.exists())

if out_jsonl.exists():
    lines = [l for l in out_jsonl.read_text(encoding='utf-8').splitlines() if l.strip()]
    print(f'reports_sections.jsonl: {len(lines)} records already parsed')
    rec = json.loads(lines[0])
    print(f'  Sample doc_id : {rec["doc_id"]}')
    print(f'  Full_text len : {len(rec["full_text"])} chars')
    for sec in SECTION_TITLES:
        n = len(rec['sections'].get(sec, ''))
        print(f'    {sec}: {n} chars')
else:
    print('Not found — run the script:')
    print('  subprocess.run([sys.executable, str(script_02)], check=True, cwd=str(BASE_DIR))')

# Uncomment to run:
# subprocess.run([sys.executable, str(script_02)], check=True, cwd=str(BASE_DIR))


---
## Step 3 — Build SFT Dataset

**Script:** `03_make_sft_dataset.py`  
**Input:** `data/reports_sections.jsonl` + `seed/*.txt`  
**Output:** `data/sft_train.jsonl`, `data/sft_valid.jsonl`

Aligns seed filenames to `doc_id` (exact → safe-filename → loose match).


In [ ]:
script_03  = BASE_DIR / '03_make_sft_dataset.py'
train_file = DATA_DIR / 'sft_train.jsonl'
valid_file = DATA_DIR / 'sft_valid.jsonl'

print('Script exists:', script_03.exists())
print()
for f in [train_file, valid_file]:
    if f.exists():
        n = len([l for l in f.read_text(encoding='utf-8').splitlines() if l.strip()])
        print(f'  {f.name}: {n} samples')
    else:
        print(f'  {f.name}: not found')

# Uncomment to run:
# subprocess.run([sys.executable, str(script_03)], check=True, cwd=str(BASE_DIR))


---
## Step 4 — QLoRA Fine-Tuning *(Skip — checkpoint already exists)*

**Script:** `04_train_sft_qwen_lora.py`

| Parameter | Value |
|-----------|-------|
| Base model | Qwen2.5-7B-Instruct |
| LoRA rank / alpha | 16 / 32 |
| Learning rate | 2e-5 |
| Epochs | 4 |
| Batch size | 1 (grad accum = 4) |
| Max seq len | 4096 tokens |
| Retrieval | BGE bge-base-zh-v1.5 + MMR |
| Evidence budget | 3600 tokens |


In [ ]:
print('=== Checkpoint Status ===')
print(f'LORA_DIR: {LORA_DIR}')
if LORA_DIR.exists():
    files = [f.name for f in LORA_DIR.iterdir()]
    print(f'Files ({len(files)}):', files)
    print('Checkpoint is ready — training can be skipped.')
else:
    print('WARNING: LoRA directory not found!')
    print('Run 04_train_sft_qwen_lora.py to train from scratch.')

# To retrain from scratch (takes several hours on RTX 3090):
# subprocess.run([sys.executable, str(BASE_DIR / '04_train_sft_qwen_lora.py')],
#                check=True, cwd=str(BASE_DIR))


---
## Step 5 — Batch Inference

**Script:** `05_batch_infer_mainidea.py`  
**Input:** `.docx` files in `data/`  
**Output:** `.txt` abstracts in `result/`

Per-document process:
1. Extract text (XML fallback for malformed files)
2. Chunk → BGE embed (cached) → Top-16 recall → MMR re-rank
3. Generate each of the 5 sections independently (max 300 chars each)
4. QC check; retry up to 2x on failure
5. Post-process and write `.txt`; log QC to `result/run_report.jsonl`


In [ ]:
import torch
print('GPU ready:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

print()
result_files = list(RESULT_DIR.glob('*.txt')) if RESULT_DIR.exists() else []
print(f'Existing result files: {len(result_files)}')

# Run batch inference:
# subprocess.run([sys.executable, str(BASE_DIR / '05_batch_infer_mainidea.py')],
#                check=True, cwd=str(BASE_DIR))


In [ ]:
# Inspect results
import json

result_files = sorted(RESULT_DIR.glob('*.txt')) if RESULT_DIR.exists() else []
print(f'Output .txt files: {len(result_files)}')

if result_files:
    sample = result_files[0]
    content = sample.read_text(encoding='utf-8')
    print(f'\n=== {sample.name} ===')
    for line in content.split('\n'):
        for sec in SECTION_TITLES:
            if line.startswith(sec + '：'):
                print(f'[{sec}] {line[len(sec)+1:].strip()[:120]}...')

# QC report
report = RESULT_DIR / 'run_report.jsonl'
if report.exists():
    records = [json.loads(l) for l in report.read_text(encoding='utf-8').splitlines() if l.strip()]
    n  = len(records)
    ok = sum(1 for r in records if r.get('qc_ok'))
    print(f'\nQC Report: {n} docs | pass: {ok} ({100*ok//n}%) | fail: {n-ok}')


---
## Example Outputs

Anonymised sample outputs from `example/` — these will be uploaded to GitHub.


In [ ]:
example_files = sorted(EXAMPLE_DIR.glob('*.txt')) if EXAMPLE_DIR.exists() else []
print(f'Example files: {len(example_files)}')
for ef in example_files:
    content = ef.read_text(encoding='utf-8')
    print(f'\n{"="*60}')
    print(f'FILE: {ef.name}  ({len(content)} chars)')
    print('='*60)
    print(content[:500])


---
## GitHub Upload

This section:
1. Writes `.gitignore` and `README.md` to the project root
2. Initialises git and makes the first commit
3. Pushes to `github.com/lofophil/edu-report-summarizer`

> **Before running:** Create an empty repo at https://github.com/new  
> Name: `edu-report-summarizer` | Public | **do NOT** add README/gitignore


### 8a. Write `.gitignore`


In [ ]:
gitignore_content = '''# Python
__pycache__/
*.py[cod]
.venv/
venv/

# Model weights (host on HuggingFace instead)
checkpoints/
*.bin
*.safetensors
*.pt
*.ckpt

# Embedding caches
**/cache_embeddings/
*.npz

# Raw data and seeds (may contain personal info)
data/reports_sections.jsonl
data/sft_train.jsonl
data/sft_valid.jsonl
seed/
研究报告/

# Results log
result/run_report.jsonl
result/

# Conversion log
convert_failures.csv

# IDE / OS
.idea/
.vscode/settings.json
.DS_Store
Thumbs.db
'''

gitignore_path = BASE_DIR / '.gitignore'
gitignore_path.write_text(gitignore_content, encoding='utf-8')
print('Written:', gitignore_path)


### 8b. Write `README.md`


In [ ]:
readme_path = BASE_DIR / 'README.md'
readme_content = "# EduReport-Summarizer\n\n**A QLoRA fine-tuned model for structured abstract extraction from Chinese educational research reports.**\n\n> Base model: `Qwen2.5-7B-Instruct` | Adapter: LoRA via PEFT  \n> Checkpoint: `qwen2.5-mainidea-final-stable-fast-infer5-seedguide`\n\n---\n\n## Project Description\n\nEduReport-Summarizer is an end-to-end pipeline for automatically extracting structured abstracts from Chinese educational research reports. Given a raw `.doc`, `.docx`, or `.pdf` report as input, the system produces a formal five-section abstract covering Research Question, Core Concepts, Research Objectives and Content, Research Outcomes, and Research Impact, with each section capped at 300 Chinese characters and the full output not exceeding 1,500 characters.\n\nThe pipeline consists of five stages. First, source documents are batch-converted to `.docx` using Microsoft Word COM automation with LibreOffice and `pdf2docx` as fallbacks. Second, each document is parsed into the five canonical sections by matching paragraph headings against an alias table, and the full text is retained for downstream use. Third, human-written gold-standard seed summaries are aligned to their corresponding documents by `doc_id` and assembled into a ChatML-format supervised fine-tuning dataset. Fourth, `Qwen2.5-7B-Instruct` is fine-tuned with QLoRA (rank 16, alpha 32) over four epochs using a retrieval-augmented evidence construction strategy: document chunks are embedded with `BAAI/bge-base-zh-v1.5`, Top-16 candidates are recalled by cosine similarity, and six diverse evidence blocks are selected per section via Maximal Marginal Relevance re-ranking within a 3,600-token budget. Fifth, the trained model runs batch inference section-by-section, with a rule-based quality-control layer that checks character limits, forbidden characters, and quotation marks, retrying generation up to twice on failure before logging the QC result.\n\nThe model strictly forbids English text, Arabic numerals, direct quotation from the source, and fabrication of information not present in the original document, making it suitable for formal educational research evaluation contexts.\n\n\n---\n\n## Pipeline\n\n```\nRaw .doc / .pdf reports\n        \u2502\n        \u25bc\n[01] 01_convert_to_docx.py       Batch convert \u2192 .docx\n        \u2502\n        \u25bc\n[02] 02_docx_to_sections.py      Parse \u2192 5-section JSONL\n        \u2502\n        \u25bc\n[03] 03_make_sft_dataset.py      Align seeds \u2192 SFT train/valid JSONL\n        \u2502\n        \u25bc\n[04] 04_train_sft_qwen_lora.py   QLoRA fine-tuning\n        \u2502\n        \u25bc\n[05] 05_batch_infer_mainidea.py  Batch inference \u2192 .txt abstracts\n```\n\n---\n\n## Project Structure\n\n```\nPythonProject/\n\u251c\u2500\u2500 01_convert_to_docx.py\n\u251c\u2500\u2500 02_docx_to_sections.py\n\u251c\u2500\u2500 03_make_sft_dataset.py\n\u251c\u2500\u2500 04_train_sft_qwen_lora.py\n\u251c\u2500\u2500 05_batch_infer_mainidea.py\n\u251c\u2500\u2500 pipeline_walkthrough.ipynb   \u2190 interactive walkthrough\n\u251c\u2500\u2500 config.py                    \u2190 centralised path & hyperparameter config\n\u251c\u2500\u2500 requirements.txt\n\u251c\u2500\u2500 README.md\n\u251c\u2500\u2500 example/                     \u2190 anonymised sample outputs\n\u251c\u2500\u2500 data/                        \u2190 converted .docx + JSONL (not uploaded)\n\u251c\u2500\u2500 seed/                        \u2190 human-written seeds (not uploaded)\n\u2514\u2500\u2500 checkpoints/                 \u2190 LoRA weights (host on HuggingFace)\n```\n\n---\n\n## Quick Start\n\n```bash\ngit clone https://github.com/lofophil/edu-report-summarizer.git\ncd edu-report-summarizer\npip install -r requirements.txt\n```\n\nDownload the base model:\n```bash\nhuggingface-cli download Qwen/Qwen2.5-7B-Instruct --local-dir ./checkpoints/base\n```\n\nPlace your LoRA weights at `checkpoints/qwen2.5-mainidea-final-stable-fast-infer5-seedguide/final_best/`\n\nRun inference:\n```bash\npython 05_batch_infer_mainidea.py\n```\n\nOr follow the notebook: `pipeline_walkthrough.ipynb`\n\n---\n\n## Output Constraints\n\nThe model is trained to enforce:\n\n- Exactly 5 sections with canonical headings\n- No English letters, Arabic numerals, or Roman numerals\n- No direct quotation from source text\n- No fabricated information\n- Max 300 characters per section; max 1,500 total\n- Formal, concise style consistent with human-written seeds\n\n---\n\n## Training Details\n\n| Hyperparameter | Value |\n|---|---|\n| Base model | Qwen/Qwen2.5-7B-Instruct |\n| LoRA rank / alpha | 16 / 32 |\n| Learning rate | 2e-5 |\n| Epochs | 4 |\n| Batch size | 1 (grad accum = 4) |\n| Max sequence length | 4096 tokens |\n| Warmup ratio | 0.08 |\n| Retrieval model | BAAI/bge-base-zh-v1.5 |\n| Chunk size | 1100 tokens, 160 overlap |\n| MMR lambda | 0.62 |\n| Evidence budget | 3600 tokens |\n\n---\n\n## RAG Strategy\n\nFor each section, the system:\n1. Chunks the document by token count (1100 tok, 160 overlap)\n2. Embeds chunks with BGE (`bge-base-zh-v1.5`), cached to disk\n3. Recalls Top-16 chunks by cosine similarity\n4. Re-ranks with MMR (lambda=0.62) \u2192 6 diverse evidence blocks\n5. Clips to 3600-token evidence budget\n6. Generates each section independently (up to 2 retries on QC failure)\n\nIf a human seed exists for the document, it enriches the retrieval query.\n\n---\n\n## Example Outputs\n\nSee [`example/`](example/) for anonymised sample outputs.\n\n---\n\n## Requirements\n\n```\ntorch>=2.0 (CUDA build)\ntransformers>=4.40\npeft>=0.10\naccelerate>=0.27\nsentence-transformers>=2.6\npython-docx\npdf2docx\nnumpy\ntqdm\n```\n\n---\n\n## Citation\n\nIf you use this project in your research or work, please cite it as:\n\n```bibtex\n@misc{lofophil2025edureport,\n  author       = {lofophil},\n  title        = {EduReport-Summarizer: Structured Abstract Extraction\n                  from Chinese Educational Research Reports via QLoRA},\n  year         = {2025},\n  publisher    = {GitHub},\n  howpublished = {\\url{https://github.com/lofophil/edu-report-summarizer}},\n  note         = {Contact: lofocdut@gmail.com}\n}\n```\n\nOr in plain text:\n\n> lofophil. (2025). *EduReport-Summarizer: Structured Abstract Extraction from Chinese Educational Research Reports via QLoRA*. GitHub. https://github.com/lofophil/edu-report-summarizer. Contact: lofocdut@gmail.com\n\n---\n\n## License\n\nMIT License\n\n---\n\n## Acknowledgements\n\n- [Qwen2.5](https://github.com/QwenLM/Qwen2.5) by Alibaba Cloud\n- [BGE](https://github.com/FlagAI-Open/FlagEmbedding) by BAAI\n- [PEFT](https://github.com/huggingface/peft) by Hugging Face\n"
readme_path.write_text(readme_content, encoding='utf-8')
print('Written:', readme_path)
print(f'README size: {len(readme_content)} chars')


### 8c. Initialise Git and Commit

> Make sure you have already created the empty repo at https://github.com/new


In [ ]:
import subprocess

def git(args, cwd=BASE_DIR):
    result = subprocess.run(
        ['git'] + args, cwd=str(cwd),
        capture_output=True, text=True, encoding='utf-8'
    )
    out = (result.stdout + result.stderr).strip()
    if out:
        print(out)
    return result.returncode

# Init
print('--- git init ---')
git(['init'])

# Config identity
git(['config', 'user.name', 'lofophil'])
git(['config', 'user.email', 'lofocdut@gmail.com'])

print('\n--- git status ---')
git(['status'])


In [ ]:
# Stage all (respects .gitignore)
print('--- git add . ---')
git(['add', '.'])

print('\n--- Staged files ---')
git(['diff', '--cached', '--name-only'])


In [ ]:
# Confirm what will be committed — check NO seed/ or checkpoints/ appear
print('Files to be committed:')
git(['diff', '--cached', '--name-status'])


In [ ]:
# First commit
print('--- git commit ---')
git(['commit', '-m', 'Initial release: EduReport-Summarizer pipeline'])


### 8d. Add Remote and Push

Replace the URL below with your actual GitHub repo URL if different.


In [ ]:
REMOTE_URL = 'https://github.com/lofophil/edu-report-summarizer.git'

print('--- add remote ---')
# Remove existing remote if any
git(['remote', 'remove', 'origin'])
git(['remote', 'add', 'origin', REMOTE_URL])

print('\n--- rename branch to main ---')
git(['branch', '-M', 'main'])

print('\n--- git push ---')
print('A browser window may open for GitHub authentication.')
git(['push', '-u', 'origin', 'main'])


### Done!

After the push completes, visit:  
**https://github.com/lofophil/edu-report-summarizer**

You should see:
- All 5 pipeline scripts
- `pipeline_walkthrough.ipynb`
- `README.md` rendered on the homepage
- `example/*.txt` sample outputs

**NOT uploaded** (excluded by `.gitignore`):
- `seed/` — human-written summaries
- `checkpoints/` — model weights
- `data/*.jsonl` — processed data
- `result/` — inference outputs
